# 00 — Universal exact-teacher dataset

Local kernel: select `streamlined_experiment/.venv-local/Scripts/python.exe`. Prompts remain unchanged. Teacher object responses retain their lexical JSON substring from the source; the original API serialization is unavailable. No model/API inference occurs here. A tokenizer-only warning about missing PyTorch is expected in this lightweight local environment.

## 1. Local setup

Locate the experiment and import shared helpers.

**Recovery / safe rerun:** After a local kernel restart, rerun this cell, then the interrupted section.

**Cost:** No paid API requests.



In [11]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

Experiment: c:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment
Interpreter: c:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Scripts\python.exe


## 2. Extract A/B/C and audit source

Extract 990 designated teacher rows; reuse the original scenario-step split. Checks produce audit data and never edit prompts or answers. Identity mismatches halt preparation to prevent incorrect joins.

**Recovery / safe rerun:** Rerun 1 then 2. Repeated extraction is identical; changed source content requires a new experiment.

**Cost:** No paid API requests.



In [12]:
from streamlined.data import prepare
source = ROOT/'data/source/llm_responses.json'
result = prepare(ROOT, source, ROOT/'data/source/group_split.json', ROOT/'data/source/llm_queries.json')
print(result)
print('Audit:', ROOT/'data/dataset_audit.csv')

{'source_sha256': 'aaa65e2a8acdd897ef14f6c44074c2f0ebdf746c7d3d4edc1372421567922f52', 'teacher': 'openai/gpt-5.5-20260423', 'first_row': 1981, 'last_row': 2970, 'rows': 990, 'split_counts': {'train': 705, 'test': 145, 'validation': 140}, 'original_split_sha256': '3c35ad2ee99aff6c976ee4a964c6d4f56a3d2873bbfe62332efde902f1598e7b', 'source_representation_note': 'Object responses preserve original source JSON substring; API-original serialization unavailable.', 'prompts_sha256': 'd96d7102456f3742e81b70dcbdbdd4a082fbdac644e5c5951f36a9131711c490', 'teachers_sha256': '259ffcf5020ed820e48c4b28056d95728be21ee1312eacaaced20567f57f4a53'}
Audit: c:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\data\dataset_audit.csv


## 3. Freeze model revisions and tokenizers

Read the HF token from the parent project secret/HF_Token.txt. **Tokenizer downloads are cached locally and reusable. No GPU is needed.** The Llama model requires approved HF access.

**Recovery / safe rerun:** After download/access errors, rerun 1 then 3. Already frozen commits remain fixed.

**Cost:** No paid API requests.



In [13]:
from streamlined.data import freeze_tokenizers
specs = freeze_tokenizers(ROOT)
print({k:{'revision':v['revision'],'context':v['context'],'training_context':v['training_context']} for k,v in specs.items()})

{'qwen1b': {'revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306', 'context': 32768, 'training_context': 8192}, 'hermes': {'revision': '898cd6ff3637a762c9b74309a384cc5db68fdfee', 'context': 32768, 'training_context': 8192}, 'bitnet2b': {'revision': '276681394656abdadb8e80e5b2c3db5e5d7fcaff', 'context': 4096, 'training_context': 4096}, 'llama3b': {'revision': '0cb88a4f764b7a12671c53f0838cd831a0843b95', 'context': 131072, 'training_context': 8192}, 'mistral7b': {'revision': 'c170c708c41dac9275d15a8fff4eca08d52bab71', 'context': 32768, 'training_context': 8192}, 'qwen7b': {'revision': 'a09a35458c702b33eeacc393d103063234e8bc28', 'context': 32768, 'training_context': 8192}}


## 4. Context feasibility and universal training exclusions

Count exact complete chats for all six tokenizers. Training uses the common set that fits every model. No prompt shortening or answer repair. Every inference row remains represented even if a context is unsupported.

**Recovery / safe rerun:** Rerun 1 then 4 after section 3 completes. No training is performed.

**Cost:** No paid API requests.



In [14]:
from streamlined.data import feasibility
print(feasibility(ROOT))
print('Exclusions:', ROOT/'data/training_exclusions.csv')

{'eligible_train': 604, 'excluded_train': 101, 'rows_audited': 5940}
Exclusions: c:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\data\training_exclusions.csv


## 5. Prepare Colab bundle and check audible monitoring

The receiver serves the prepared code/data and cached tokenizers. API credentials are excluded; HF credentials are supplied only to authenticated Colab workers in memory.

**Recovery / safe rerun:** Start the terminals below, then rerun 1 and 5. After a tunnel restart, reconnect each Colab notebook.

**Cost:** No paid API requests.

**Laptop PowerShell commands (separate terminals; run once for all Colab notebooks):**

Terminal 1 — local receiver:
```powershell
Set-Location 'C:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
.\.venv-local\Scripts\python.exe -m streamlined.receiver --root . --port 8770
```
Terminal 2 — HTTPS tunnel:
```powershell
Set-Location 'C:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
cloudflared tunnel --url http://127.0.0.1:8770
```
If `cloudflared` is unavailable: `winget install --id Cloudflare.cloudflared --exact`, then reopen the terminal.
Paste the printed `https://…trycloudflare.com` URL in the connection cell. Keep this terminal open.

Terminal 3 — independent audible watchdog:
```powershell
Set-Location 'C:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
.\.venv-local\Scripts\python.exe -m streamlined.monitor --root .
```
Terminal 4 — read the receiver token, test sound, or acknowledge an alarm:
```powershell
Set-Location 'C:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment'
Get-Content .\runtime\receiver_token.txt
.\.venv-local\Scripts\python.exe -m streamlined.monitor --root . --test
# Run only to acknowledge an active alarm:
# .\.venv-local\Scripts\python.exe -m streamlined.monitor --root . --ack
```
The token is generated by Terminal 1, is distinct from API/HF credentials, and stays in ignored `runtime/`.
All notebooks may share the URL/token; run each model/condition only once at a time. Use separate Colab runtimes for parallel models.
Keep Windows awake and audio unmuted; test the alarm before a long run. Loss of heartbeats alarms after 150 seconds.


In [9]:
from streamlined.transport import make_bundle
from streamlined.notebook import watchdog_ready
watchdog_ready(ROOT)
assert (ROOT/'manifests/experiment.json').is_file(), 'Complete section 4 first'
print('Bundle:', make_bundle(ROOT))

Bundle: c:\Users\nelptp2nmap\Documents\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\runtime\bootstrap.zip


## 6. Optional local base-model cache

**Full base weights can be downloaded once on this laptop without a GPU.** This avoids spending Colab time downloading them from Hugging Face. Archives plus extracted weights require substantial disk space (roughly twice the checkpoint sizes). It is optional: Colab otherwise downloads and mirrors them.

**Recovery / safe rerun:** Rerun 1 and 6 after a download interruption. Hugging Face reuses downloaded files. No paid requests or model execution.

**Cost:** Network transfer and local disk use; no GPU needed.



In [ ]:
DOWNLOAD_BASES_LOCALLY = False
SELECTED_CACHE_MODELS = list(MODELS)
if DOWNLOAD_BASES_LOCALLY:
    from streamlined.data import cache_bases
    cache_bases(ROOT,SELECTED_CACHE_MODELS)